<a href="https://colab.research.google.com/github/igd1990/hugging-face-tasks/blob/tiny-hidream-i1-pipe/tiny_hidream_i1_pipe.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -U diffusers

## Local Inference on GPU
Model page: https://huggingface.co/hf-internal-testing/tiny-hidream-i1-pipe

⚠️ If the generated code snippets do not work, please open an issue on either the [model repo](https://huggingface.co/hf-internal-testing/tiny-hidream-i1-pipe)
			and/or on [huggingface.js](https://github.com/huggingface/huggingface.js/blob/main/packages/tasks/src/model-libraries-snippets.ts) 🙏

In [ ]:
pip install -U diffusers transformers accelerate

In [ ]:
from diffusers import DiffusionPipeline
from transformers import CLIPProcessor, CLIPModel
import torch, time

# 1. Load diffusion model
pipe = DiffusionPipeline.from_pretrained(
    "hf-internal-testing/tiny-hidream-i1-pipe"
).to("cuda" if torch.cuda.is_available() else "cpu")

# 2. Load CLIP for evaluation
clip_model_name = "openai/clip-vit-base-patch32"
clip_model = CLIPModel.from_pretrained(clip_model_name).eval()
clip_processor = CLIPProcessor.from_pretrained(clip_model_name)

device = "cuda" if torch.cuda.is_available() else "cpu"
clip_model.to(device)

prompts = [
    "a red apple",
    "a yellow train",
    "a robot standing on railroad tracks",
    # ...
]

results = []

for prompt in prompts:
    start = time.time()
    image = pipe(prompt).images[0]
    latency = time.time() - start

    # CLIP score
    inputs = clip_processor(text=[prompt], images=[image], return_tensors="pt", padding=True).to(device)
    with torch.no_grad():
        out = clip_model(**inputs)
        # CLIP uses cosine sim between pooled embeddings
        img_emb = out.image_embeds / out.image_embeds.norm(p=2, dim=-1, keepdim=True)
        txt_emb = out.text_embeds  / out.text_embeds.norm(p=2, dim=-1, keepdim=True)
        clip_score = (img_emb * txt_emb).sum(-1).item()

    results.append({
        "prompt": prompt,
        "clip_score": clip_score,
        "latency_sec": latency,
    })


In [ ]:
import pandas as pd

df = pd.DataFrame(results)
print(df.sort_values("clip_score", ascending=False))

print("Mean CLIP score:", df["clip_score"].mean())
print("Mean latency (s):", df["latency_sec"].mean())
